# ДЗ 11. AI-агенти на Agno — кейс ChefBot

**Виконала:** Команіч Надія Миколаївна
**Варіант:** ускладнений (завдання 1–5)
**Кейс:** ChefBot — кулінарний асистент: пошук рецептів, конвертація кухонних мір, підбір замінників інгредієнтів.

Усі агенти працюють на Google Gemini через фреймворк Agno. Запускайте комірки послідовно згори донизу (Shift+Enter).

> Продуктові коментарі написано за результатами фактичного запуску ноутбука (модель gemini-3.5-flash-lite).

## Підготовка середовища

In [ ]:
# Встановлюємо залежності у версіях, зазначених в умові
%pip install --quiet "agno==3.1.12" "ddgs==9.16.1" "google-genai==2.23.0" "pytz==2026.6"

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.2 which is incompatible.


Якщо після встановлення з'являються помилки імпорту — **Runtime → Restart session** і запустіть комірки знову з першої.

In [ ]:
import os
import getpass

# Беремо ключ із Colab Secrets (іконка 🔑 зліва, секрет GOOGLE_API_KEY)
try:
    from google.colab import userdata
    api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    api_key = None

# Якщо секрету немає — просимо ввести ключ вручну (у код його не вписуємо)
if not api_key:
    api_key = getpass.getpass("Введіть Google Gemini API key: ")

if not api_key:
    raise ValueError("API-ключ не задано.")

os.environ["GOOGLE_API_KEY"] = api_key
print("✅ API-ключ налаштовано")

✅ API-ключ налаштовано


In [ ]:
import time
from agno.agent import Agent
from agno.models.google import Gemini
from agno.db.in_memory import InMemoryDb

# На безкоштовному тарифі gemini-3.8-flash має ліміт лише 20 запитів на день,
# а весь ноутбук робить їх набагато більше. Тому основною обрано альтернативну
# модель з умови завдання; рекомендовану можна повернути, якщо є платний тариф.
MODEL_ID = "gemini-3.5-flash-lite"
# MODEL_ID = "gemini-3.8-flash"


def make_model() -> Gemini:
    """Створює модель з автоматичними повторами при тимчасових збоях (503)."""
    return Gemini(id=MODEL_ID, retries=3, delay_between_retries=10, exponential_backoff=True)

# Швидка перевірка підключення
check_agent = Agent(
    model=make_model(),
    description="Ти привітний кулінарний асистент ChefBot.",
    markdown=True,
)
check_agent.print_response("Привіт! Скажи одним реченням, чим ти можеш допомогти на кухні.", stream=True)

Output()

In [ ]:
# Допоміжна функція: запускає агента/команду і показує час виконання
def run_timed(runner, prompt: str, pause: int = 8, **kwargs) -> float:
    time.sleep(pause)  # пауза між запитами, щоб не впертися в ліміт запитів за хвилину
    start = time.perf_counter()
    runner.print_response(prompt, stream=True, **kwargs)
    elapsed = time.perf_counter() - start
    print(f"\n⏱ Час виконання: {elapsed:.1f} с")
    return elapsed

## Завдання 1. Агент з інструментами веб-пошуку

Агент рівня L1: модель Gemini + DuckDuckGoTools. Інструкції адаптовано під ChefBot: шукати кулінарну інформацію і завжди давати посилання на джерела.

In [ ]:
from agno.tools.duckduckgo import DuckDuckGoTools
from ddgs import DDGS

# Перевіряємо пошук напряму, без моделі. У першому запуску сервер DuckDuckGo
# не віддавав результатів для Colab («No results found»), тому вмикаємо
# backend="auto": бібліотека пробує кілька пошукових рушіїв по черзі.
try:
    probe = DDGS().text("chicken safe internal temperature", max_results=3, backend="auto")
    print(f"✅ Пошук працює, знайдено результатів: {len(probe)}")
    for r in probe:
        print("  •", r.get("title"), "—", r.get("href"))
except Exception as e:
    print("⚠️ Пошук недоступний із Colab:", e)

# Створюємо агента з веб-пошуком
web_agent = Agent(
    name="ChefBot Web Research",
    model=make_model(),
    tools=[DuckDuckGoTools(backend="auto")],          # інструмент пошуку в інтернеті
    instructions=[
        "Ти кулінарний дослідник ChefBot.",
        "Відповідай українською мовою.",
        "Для актуальних питань обов'язково використовуй пошук, а не пам'ять.",
        "Завжди вказуй джерела інформації з посиланнями.",
        "Якщо джерела суперечать одне одному — прямо скажи про це.",
    ],
    markdown=True,
    stream_events=True,   # показуємо проміжні події, зокрема виклики інструментів
    debug_mode=True,      # технічні логи: промпт, аргументи викликів, токени
)

✅ Пошук працює, знайдено результатів: 3
  • Doneness of chicken — https://grokipedia.com/page/Doneness_of_chicken
  • Stop Overcooking Chicken: The Safer, Juicier Way to Get It Done — https://www.seriouseats.com/safe-chicken-temperature-time-and-temp-11948586
  • Cook to a Safe Minimum Internal Temperature - FoodSafety.govChicken Internal Temperature: 165°F by CutChicken Internal Temp Guide (Safe Cooking Temperatures)Chicken Internal Temperature Chart: Safe Temps for Every CutSafe Minimum Internal Temperature Chart - Food Safety and ...Chicken Internal Temperature Chart (Breast, Thigh, Wings ... — https://www.foodsafety.gov/food-safety-charts/safe-minimum-internal-temperatures


### Запит 1 — актуальні новини

In [ ]:
t1_news = run_timed(web_agent, "Які гастрономічні тренди та новини ресторанної індустрії обговорюють цього місяця? Дай 3–5 пунктів.")

DEBUG Agent initialized: chefbot-web-research

DEBUG Creating new AgentSession: c3c1f435-a3c6-4f5b-b87d-ae01b54e1e88

DEBUG Agent Run Start: ac36e4fc-4761-4004-99a9-3be44a100970

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

DEBUG Added tool search_news from websearch

... (41 outputs left out)

DEBUG Google Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: ac36e4fc-4761-4004-99a9-3be44a100970


⏱ Час виконання: 5.6 с


### Запит 2 — пошук фактів

In [ ]:
t1_fact = run_timed(web_agent, "За якої внутрішньої температури куряче філе вважається безпечним для споживання? Наведи рекомендацію харчових регуляторів.")

DEBUG Session ID: c3c1f435-a3c6-4f5b-b87d-ae01b54e1e88

DEBUG Agent initialized: chefbot-web-research

DEBUG Creating new AgentSession: c3c1f435-a3c6-4f5b-b87d-ae01b54e1e88

DEBUG Agent Run Start: ec5e92fe-67d9-4d21-82b5-a939f0725341

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (58 outputs left out)

DEBUG Google Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: ec5e92fe-67d9-4d21-82b5-a939f0725341


⏱ Час виконання: 9.3 с


### Запит 3 — факти про продукт

In [ ]:
t1_fact2 = run_timed(web_agent, "Чим відрізняється борошно вищого ґатунку від борошна з маркуванням W 00 для піци? Яке краще для неаполітанського тіста?")

DEBUG Session ID: c3c1f435-a3c6-4f5b-b87d-ae01b54e1e88

DEBUG Agent initialized: chefbot-web-research

DEBUG Creating new AgentSession: c3c1f435-a3c6-4f5b-b87d-ae01b54e1e88

DEBUG Agent Run Start: 028a2c97-1c04-48d2-a0f8-26e8c33360e7

DEBUG Processing tools for model

DEBUG Added tool web_search from websearch

... (87 outputs left out)

DEBUG Google Response Stream End

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 028a2c97-1c04-48d2-a0f8-26e8c33360e7


⏱ Час виконання: 14.9 с


### Порівняння: той самий запит без інструменту

In [ ]:
# Контрольний агент без пошуку — для відповіді на перше питання продуктового коментаря
plain_agent = Agent(
    name="ChefBot без пошуку",
    model=make_model(),
    instructions=["Відповідай українською мовою.", "Вказуй джерела, якщо можеш."],
    markdown=True,
)
t1_plain = run_timed(plain_agent, "Які гастрономічні тренди та новини ресторанної індустрії обговорюють цього місяця? Дай 3–5 пунктів.")

DEBUG Session ID: d4ab53f0-91f6-46c1-99ae-2bff68104ce4

Output()


⏱ Час виконання: 3.6 с


### 📝 Продуктовий коментар до завдання 1

**Чим відповідь з DuckDuckGo відрізняється від відповіді Gemini без інструменту?**
На запиті про новини різниця принципова. Агент без інструменту (3,6 с) видав «вічнозелені» тренди: upcycling і zero waste, локальні продукти, безалкогольні коктейлі. Він подав їх як новини «цього місяця», а замість джерел назвав видання (Eater, Національна ресторанна асоціація) без жодного посилання. Агент з пошуком (5,6 с) двічі викликав `search_news` і назвав конкретні речі: скорочення розриву цін між їжею вдома і в ресторані, рейтинги Yelp з бранчами й стейк-хаусами, те, що лише 20–30 % тимчасових пропозицій залишаються в меню. До кожного пункту є посилання на статтю (FastCasual, New York Post, Convenience Store News).
На стабільному факті (температура готовності курки) обидва підходи дають 74 °C: модель знає це і без пошуку. Пошук тут додає лише посилання, і вони виявилися слабким місцем (див. нижче).
Висновок: пошук виграє там, де потрібні **свіжість і конкретика**. Для стабільних фактів він майже не змінює зміст.

**Ризики, які я побачила на власних запусках.**
1. *Неправильна атрибуція.* У запиті 2 агент написав «джерела: USDA / ienergo.com.ua» і послався на gelios.ua та zemelka.ua. Це сайти енергокомпанії й інтернет-магазинів, а не регулятор. Висновок правильний, але посилання на USDA підкріплено чужою сторінкою.
2. *Якість джерел.* У запиті 3 з 13 посилань більшість ведуть на магазини й блоги (prom.ua, zapodarkom, smakotti), а не на технологів чи стандарти.
3. *Нестабільний пошук.* У запиті 2 два з трьох викликів, у запиті 3 три з п'яти повернули «No results found». У попередньому запуску цього ж ноутбука пошук із Colab не спрацював **жодного разу**, а агент однаково дописав розділ «Джерела» з посиланнями на USDA і ДСТУ. Тобто він «процитував» те, чого не читав.

**Що дають PM `stream_events=True` і `debug_mode=True`?**
Без цих режимів усі три ризики вище були б невидимими: у фінальному тексті впевнений тон, «джерела» і посилання. Лише в журналі подій видно, скільки пошуків було, які повернули порожній результат і що саме знайдено. Для оцінки якості це спосіб знайти причину помилки: поганий пошуковий запит, порожня видача або неправильний переказ знайденого. Кожна причина виправляється по-своєму. Без логів PM оцінює лише стиль тексту, а не те, як відповідь отримано. До того ж логи показують модель, токени й час, а це вартість і швидкість продукту.

**Де пошук в інтернеті додає ChefBot цінність, а де ризик?**
*Цінність:* сезонні й трендові страви, рецепти конкретних шефів, рідкісні національні страви, яких немає в базі, і актуальні рекомендації регуляторів щодо безпеки продуктів.
*Ризик:* модель може процитувати джерело, якого не читала, або приписати факт не тому сайту. Кулінарні блоги й магазини містять помилки в пропорціях і небезпечні поради, наприклад консервацію без стерилізації. Видача нестабільна. Тому для безпекових тем (алергени, температури, консервація) ChefBot має спиратися на перевірену базу, а веб-пошук залишити для натхнення. Також потрібне правило: не показувати розділ «Джерела», якщо пошук нічого не повернув.

## Завдання 2. Власні інструменти

1. `get_current_datetime` — поточна дата і час у заданому часовому поясі.
2. `convert_kitchen_measure` — конвертація кухонних мір **з урахуванням інгредієнта**: склянка борошна і склянка цукру важать по-різному, тож модель «з пам'яті» тут часто помиляється.

In [ ]:
from agno.tools import tool


@tool
def get_current_datetime(timezone: str = "Europe/Kyiv") -> str:
    """Повертає поточну дату, день тижня і час у вказаному часовому поясі.

    Використовуй цей інструмент, коли користувач питає, котра година, яка сьогодні
    дата чи день тижня, або коли для відповіді потрібно знати «зараз»
    (наприклад, «що приготувати на вечерю сьогодні», «скільки лишилось до свята»).
    НЕ використовуй його для питань про рецепти чи міри.

    Args:
        timezone: Назва часового поясу у форматі IANA, наприклад "Europe/Kyiv",
            "Europe/Warsaw", "America/New_York". За замовчуванням "Europe/Kyiv".

    Returns:
        Рядок з датою і часом у форматі ISO 8601 та днем тижня
        або повідомлення про помилку, якщо часовий пояс невідомий.
    """
    from datetime import datetime
    import pytz

    try:
        tz = pytz.timezone(timezone)
    except pytz.UnknownTimeZoneError:
        return (f"Помилка: невідомий часовий пояс '{timezone}'. "
                "Використай формат IANA, наприклад 'Europe/Kyiv'.")

    now = datetime.now(tz)
    days = ["понеділок", "вівторок", "середа", "четвер", "п'ятниця", "субота", "неділя"]
    return (f"Поточна дата та час ({timezone}): {now.isoformat(timespec='seconds')}, "
            f"{days[now.weekday()]}")


# Щільність інгредієнтів: скільки грамів в 1 мл (усереднені кухонні значення)
DENSITY_G_PER_ML = {
    "вода": 1.0, "молоко": 1.03, "вершки": 1.0, "олія": 0.92,
    "борошно": 0.53, "цукор": 0.85, "цукрова пудра": 0.56, "сіль": 1.2,
    "мед": 1.42, "рис": 0.8, "вівсянка": 0.38, "какао": 0.42,
    "масло вершкове": 0.96, "крохмаль": 0.6, "манка": 0.67,
}

# Об'ємні одиниці в мілілітрах (українська кухонна традиція: склянка = 250 мл)
VOLUME_ML = {
    "мл": 1.0, "л": 1000.0,
    "склянка": 250.0, "чашка": 240.0,      # чашка — американська cup
    "ст. л.": 15.0, "ч. л.": 5.0,
}
WEIGHT_G = {"г": 1.0, "кг": 1000.0}

# Синоніми, щоб інструмент розумів різні написання
UNIT_ALIASES = {
    "мілілітр": "мл", "ml": "мл", "літр": "л", "l": "л",
    "склянки": "склянка", "склянок": "склянка", "cup": "чашка", "cups": "чашка",
    "столова ложка": "ст. л.", "ст.л.": "ст. л.", "ст л": "ст. л.", "tbsp": "ст. л.",
    "чайна ложка": "ч. л.", "ч.л.": "ч. л.", "ч л": "ч. л.", "tsp": "ч. л.",
    "грам": "г", "грами": "г", "грамів": "г", "g": "г",
    "кілограм": "кг", "kg": "кг",
}


def _norm_unit(unit: str) -> str:
    u = unit.strip().lower()
    return UNIT_ALIASES.get(u, u)


@tool
def convert_kitchen_measure(amount: float, from_unit: str, to_unit: str, ingredient: str = "вода") -> str:
    """Точно конвертує кухонні міри з урахуванням щільності інгредієнта.

    Використовуй цей інструмент ЗАВЖДИ, коли користувач просить перевести кількість
    інгредієнта з однієї одиниці в іншу: склянки, чашки (cup), столові та чайні ложки,
    мл, л, г, кг. Особливо коли треба перевести об'єм у вагу або навпаки —
    результат залежить від інгредієнта (склянка борошна ≈ 130 г, а цукру ≈ 210 г),
    тому не рахуй це з пам'яті.

    Args:
        amount: Кількість у вихідній одиниці, число більше нуля (наприклад, 1.5).
        from_unit: Вихідна одиниця: "мл", "л", "склянка" (250 мл), "чашка" (US cup, 240 мл),
            "ст. л.", "ч. л.", "г", "кг".
        to_unit: Цільова одиниця з того самого списку.
        ingredient: Назва інгредієнта українською в називному відмінку
            ("борошно", "цукор", "молоко"...). Потрібна для переведення об'єм ↔ вага.

    Returns:
        Рядок з результатом конвертації та використаною щільністю
        або зрозуміле повідомлення про помилку (невідома одиниця чи інгредієнт).
    """
    if amount is None or amount <= 0:
        return "Помилка: кількість має бути додатним числом."

    fu, tu = _norm_unit(from_unit), _norm_unit(to_unit)
    known = set(VOLUME_ML) | set(WEIGHT_G)
    for u, raw in ((fu, from_unit), (tu, to_unit)):
        if u not in known:
            return f"Помилка: невідома одиниця '{raw}'. Доступні: {', '.join(sorted(known))}."

    # Той самий тип одиниць — щільність не потрібна
    if fu in VOLUME_ML and tu in VOLUME_ML:
        result = amount * VOLUME_ML[fu] / VOLUME_ML[tu]
        return f"{amount:g} {fu} = {result:.2f} {tu} (переведення об'єму, інгредієнт не впливає)."
    if fu in WEIGHT_G and tu in WEIGHT_G:
        result = amount * WEIGHT_G[fu] / WEIGHT_G[tu]
        return f"{amount:g} {fu} = {result:.2f} {tu}."

    # Об'єм ↔ вага — потрібна щільність
    ing = ingredient.strip().lower()
    density = DENSITY_G_PER_ML.get(ing)
    if density is None:
        return (f"Немає даних про щільність інгредієнта '{ingredient}'. "
                f"Відомі: {', '.join(sorted(DENSITY_G_PER_ML))}. "
                "Порадь користувачу зважити продукт на кухонних вагах.")

    if fu in VOLUME_ML:                       # об'єм → вага
        grams = amount * VOLUME_ML[fu] * density
        result = grams / WEIGHT_G[tu]
    else:                                     # вага → об'єм
        ml = amount * WEIGHT_G[fu] / density
        result = ml / VOLUME_ML[tu]
    return (f"{amount:g} {fu} ({ing}) ≈ {result:.1f} {tu} "
            f"(щільність {density} г/мл; значення усереднене, для випічки краще зважити).")

### Перевірка інструментів без моделі (юніт-тести)

In [ ]:
# Викликаємо реальні функції напряму через .entrypoint — щоб переконатися,
# що логіка працює ще до того, як її викликатиме модель
print(get_current_datetime.entrypoint("Europe/Kyiv"))
print(get_current_datetime.entrypoint("Mars/Olympus"))
print(convert_kitchen_measure.entrypoint(2, "склянки", "г", "борошно"))
print(convert_kitchen_measure.entrypoint(300, "г", "склянка", "цукор"))
print(convert_kitchen_measure.entrypoint(3, "ст. л.", "мл", "олія"))
print(convert_kitchen_measure.entrypoint(1, "cup", "г", "кіноа"))
print(convert_kitchen_measure.entrypoint(-1, "г", "кг"))

Поточна дата та час (Europe/Kyiv): 2026-11-16T11:00:14+02:00, понеділок
Помилка: невідомий часовий пояс 'Mars/Olympus'. Використай формат IANA, наприклад 'Europe/Kyiv'.
2 склянка (борошно) ≈ 265.0 г (щільність 0.53 г/мл; значення усереднене, для випічки краще зважити).
300 г (цукор) ≈ 1.4 склянка (щільність 0.85 г/мл; значення усереднене, для випічки краще зважити).
3 ст. л. = 45.00 мл (переведення об'єму, інгредієнт не впливає).
Немає даних про щільність інгредієнта 'кіноа'. Відомі: борошно, вершки, вода, вівсянка, какао, крохмаль, манка, масло вершкове, мед, молоко, олія, рис, сіль, цукор, цукрова пудра. Порадь користувачу зважити продукт на кухонних вагах.
Помилка: кількість має бути додатним числом.


In [ ]:
custom_agent = Agent(
    name="ChefBot Custom Tools",
    model=make_model(),
    tools=[get_current_datetime, convert_kitchen_measure],
    instructions=[
        "Ти кулінарний асистент ChefBot.",
        "Для дати й часу використовуй get_current_datetime.",
        "Для будь-яких переведень мір використовуй convert_kitchen_measure, не рахуй з пам'яті.",
        "Якщо інструмент повернув помилку — поясни її користувачу і запропонуй вихід.",
        "Відповідай українською.",
    ],
    markdown=True,
    stream_events=True,
    debug_mode=True,
)

### Тест 1 — інструмент дати/часу

In [ ]:
run_timed(custom_agent, "Який сьогодні день тижня? Встигну до 19:00 спекти пиріг, якщо він печеться годину?")

DEBUG Agent initialized: chefbot-custom-tools

DEBUG Creating new AgentSession: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent Run Start: da3da50c-6772-449d-b601-19204707998d

DEBUG Processing tools for model

DEBUG Added tool get_current_datetime

DEBUG Added tool convert_kitchen_measure

... (28 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: da3da50c-6772-449d-b601-19204707998d


⏱ Час виконання: 1.6 с


1.5786426499998925

### Тест 2 — інструмент конвертації

In [ ]:
run_timed(custom_agent, "У бабусиному рецепті 2,5 склянки борошна і пів склянки цукру. Скільки це в грамах?")

DEBUG Session ID: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent initialized: chefbot-custom-tools

DEBUG Creating new AgentSession: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent Run Start: fe0b5d68-eb5f-4b85-bbac-737940423e9e

DEBUG Processing tools for model

DEBUG Added tool get_current_datetime

... (33 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: fe0b5d68-eb5f-4b85-bbac-737940423e9e


⏱ Час виконання: 4.3 с


4.318460768000023

### Тест 3 — запит на обидва інструменти одночасно

In [ ]:
run_timed(custom_agent, "Яка зараз дата у Варшаві? І переведи 200 г меду в столові ложки.")

DEBUG Session ID: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent initialized: chefbot-custom-tools

DEBUG Creating new AgentSession: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent Run Start: 7a627564-54fb-4113-a0eb-6084bcf92757

DEBUG Processing tools for model

DEBUG Added tool get_current_datetime

... (33 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 7a627564-54fb-4113-a0eb-6084bcf92757


⏱ Час виконання: 1.7 с


1.7411228739997568

### Тест 4 — запит, для якого жоден інструмент не підходить

In [ ]:
run_timed(custom_agent, "Порадь гарне вино до запеченої качки з яблуками.")

DEBUG Session ID: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent initialized: chefbot-custom-tools

DEBUG Creating new AgentSession: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent Run Start: 1fe76c78-56cd-44d5-b12a-9796b773a8a1

DEBUG Processing tools for model

DEBUG Added tool get_current_datetime

... (17 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: 1fe76c78-56cd-44d5-b12a-9796b773a8a1


⏱ Час виконання: 3.4 с


3.352072509000209

### Тест 5 — інгредієнта немає в базі

In [ ]:
run_timed(custom_agent, "Скільки грамів у склянці кіноа?")

DEBUG Session ID: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent initialized: chefbot-custom-tools

DEBUG Creating new AgentSession: afcf702e-22cf-4180-ba5c-13c40f4c25b4

DEBUG Agent Run Start: fbda1fdb-1134-42f2-87b2-4a7615914df3

DEBUG Processing tools for model

DEBUG Added tool get_current_datetime

... (29 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Agent Run End: fbda1fdb-1134-42f2-87b2-4a7615914df3


⏱ Час виконання: 1.7 с


1.6637533719999738

### 📝 Продуктовий коментар до завдання 2

**Що показав запуск.** Усі тести пройшли коректно:
- «Який сьогодні день тижня…» → `get_current_datetime(Europe/Kyiv)`, «понеділок, 11:00, до 19:00 ще 8 годин, встигнете».
- «2,5 склянки борошна і пів склянки цукру» → **два** виклики `convert_kitchen_measure`, **331 г і 106 г**. Ці числа точно збігаються з розрахунком інструмента (2,5 × 250 мл × 0,53 г/мл).
- Запит на дві дії → агент викликав **обидва** інструменти одним кроком: Варшава, 10:00, і 200 г меду ≈ 9,4 ст. л.
- Вино до качки → **жодного** виклику інструментів, відповідь із загальних знань.
- Склянка кіноа → інструмент чесно повернув «немає даних про щільність».

**Порівняння мого docstring з docstring інструментів теми 8.**
*Спільне:* у темі 8 `convert_units` мав той самий каркас (опис, Args, Returns) і так само був для моделі єдиним джерелом знань про інструмент. Звідти ж прийшли таблиця щільностей і склянка 250 мл.
*Різниця:* у темі 8 docstring коротко казав, *що* робить інструмент. Новий пояснює також, *коли* його викликати («ЗАВЖДИ, коли треба перевести одиниці»), *чому не можна рахувати з пам'яті* (склянка борошна ≈ 130 г, цукру ≈ 210 г) і перелічує допустимі одиниці. Новий текст точніше описує момент використання. Це видно з тестів 2 і 3: модель сама розклала один запит на два виклики з правильними аргументами.

**Чому конвертер має право бути окремим tool?**
Модель знає приблизні співвідношення, але не стабільно. У цьому ж ноутбуці (перевірка Analyst у завданні 3) модель «з пам'яті» назвала вагу склянки борошна діапазоном 130–160 г, а інструмент дає одне відтворюване число. Для випічки різниця у 20 % помітна. Таблицю щільностей можна перевірити й розширити, не змінюючи промпт. Це той самий поділ «модель розуміє запит, система рахує», який ми закладали в архітектуру ChefBot.

**Запит, для якого жоден інструмент не підходить.**
На питання про вино агент не викликав нічого і відповів із власних знань. Це правильна поведінка: примусовий виклик невідповідного інструменту дав би безглуздий результат. Цікавіше виявився тест 5. Інструмент повернув «немає даних про кіноа», а модель після цього **сама** додала «одна склянка більшості круп важить приблизно 170–200 г». У темі 8 (gpt-4o-mini) було так само: «приблизно 170 грамів». Отже, чесна відмова інструмента не гарантує чесної відповіді: модель заповнює прогалину оцінкою з пам'яті. Добре, що вона позначила число як приблизне. Якщо ж потрібна сувора поведінка, це треба прямо заборонити в інструкції.

## Завдання 3. Команда агентів

- **Researcher**: шукає інформацію в інтернеті (DuckDuckGo).
- **Analyst**: без інструментів, структурує знайдене у звіт.
- **Команда** в режимі `coordinate`: лідер сам вирішує, кому і що делегувати.

Запит під ChefBot: підготувати аналітичну довідку для розвитку продукту.

In [ ]:
from agno.team.team import Team
from agno.team.mode import TeamMode

TEAM_PROMPT = ("Досліди, які функції мають популярні кулінарні застосунки з AI (планування меню, "
               "розпізнавання продуктів з фото, списки покупок) і напиши короткий аналітичний звіт: "
               "що з цього варто додати в ChefBot для сімейного використання.")


def build_team(share: bool) -> Team:
    """Створює команду; параметр share керує share_member_interactions."""
    researcher = Agent(
        name="Researcher",
        role="Збір та верифікація інформації з інтернету",
        model=make_model(),
        tools=[DuckDuckGoTools(backend="auto")],
        instructions=[
            "Шукай актуальну інформацію з надійних джерел (офіційні сайти, огляди, магазини застосунків).",
            "Для кожного факту вказуй джерело з посиланням.",
            "Якщо не впевнена в факті — так і познач: [неперевірено].",
            "Відповідай українською мовою.",
        ],
        markdown=True,
    )
    analyst = Agent(
        name="Analyst",
        role="Аналіз інформації та написання структурованих звітів",
        model=make_model(),
        instructions=[
            "Аналізуй лише ту інформацію, яку отримала від дослідника.",
            "Структура: вступ, ключові знахідки, рекомендації для ChefBot, висновок.",
            "Познач факти без джерела як ризиковані.",
            "Пиши лаконічно та інформативно українською.",
        ],
        markdown=True,
    )
    return Team(
        name="ChefBot Research Team",
        mode=TeamMode.coordinate,          # лідер координує учасників
        members=[researcher, analyst],
        model=make_model(),
        instructions=[
            "1. Спочатку попроси Researcher знайти інформацію.",
            "2. Потім передай результати Analyst для аналізу та оформлення.",
            "3. Поверни структурований звіт українською мовою.",
        ],
        share_member_interactions=share,   # чи бачать учасники взаємодії одне одного
        show_members_responses=True,       # показуємо відповіді кожного учасника
        markdown=True,
        debug_mode=True,
    )


research_team = build_team(share=True)
t3_share_true = run_timed(research_team, TEAM_PROMPT)

DEBUG Function: web_search registered with websearch

DEBUG Function: search_news registered with websearch

DEBUG Team ID: chefbot-research-team

DEBUG Session ID: e45922a4-4be1-4bcb-9d7e-5b4f2758e7f8

DEBUG Creating new TeamSession: e45922a4-4be1-4bcb-9d7e-5b4f2758e7f8

DEBUG Team Run Start: e1f7d2e4-6545-4444-875f-a581f80db390

... (190 outputs left out)

DEBUG Google Response Stream End

DEBUG Added RunOutput to Team Session

DEBUG Team Run End: e1f7d2e4-6545-4444-875f-a581f80db390


⏱ Час виконання: 44.5 с


### Порівняння: `share_member_interactions=False`

In [ ]:
research_team_no_share = build_team(share=False)
t3_share_false = run_timed(research_team_no_share, TEAM_PROMPT)

print(f"share=True:  {t3_share_true:.1f} с")
print(f"share=False: {t3_share_false:.1f} с")

DEBUG Function: web_search registered with websearch

DEBUG Function: search_news registered with websearch

DEBUG Team ID: chefbot-research-team

DEBUG Session ID: d45ee3dc-b87f-4eb4-96e0-4ff3ca6a8715

DEBUG Creating new TeamSession: d45ee3dc-b87f-4eb4-96e0-4ff3ca6a8715

DEBUG Team Run Start: 4ae502a9-1108-4d76-900a-b923c49696a0

... (215 outputs left out)

DEBUG Google Response Stream End

DEBUG Added RunOutput to Team Session

DEBUG Team Run End: 4ae502a9-1108-4d76-900a-b923c49696a0


⏱ Час виконання: 33.0 с
share=True:  44.5 с
share=False: 33.0 с


### Перевірка: як Analyst реагує на неточну інформацію

In [ ]:
# Навмисно підкладаємо помилкові «факти дослідника», щоб перевірити, чи є верифікація
analyst_only = Agent(
    name="Analyst",
    model=make_model(),
    instructions=[
        "Аналізуй отриману інформацію і напиши короткий звіт українською.",
        "Познач факти без джерела як ризиковані.",
    ],
    markdown=True,
)
fake_research = (
    "Дані від Researcher:\n"
    "1. Мед можна давати дітям з 3 місяців як заміну цукру.\n"
    "2. Склянка борошна важить 300 г.\n"
    "3. Понад 90% користувачів кулінарних застосунків планують меню на тиждень наперед (джерело не вказано)."
)
analyst_only.print_response(fake_research + "\n\nНапиши рекомендації для ChefBot на основі цих даних.", stream=True)

DEBUG Session ID: f0e82a9c-4f3c-4f3f-b1fc-473d0a01adff

Output()

### 📝 Продуктовий коментар до завдання 3

**Як працювала команда.** В обох запусках лідер дотримався інструкції: спочатку `delegate_task_to_member → researcher`, потім `→ analyst`. Researcher робив серію пошуків, і приблизно половина з них повернула порожній результат (4 з 7 і 4 з 9). Фінальні звіти мають структуру «огляд функцій → що додати в ChefBot» і посилаються на ChefGPT, Mealime, SuperCook, Plant Jammer.

**Чи виправдана затримка?**
Команда відповідала **44,5 с**, а одиночний агент із завдання 1 — 3,6–14,9 с. Під капотом відбулося щонайменше чотири звернення до моделі плюс серія пошуків, а одна відповідь моделі ще й прийшла з помилкою 503 і повтором. Для мого прикладу, аналітичної довідки для розвитку продукту, затримка **виправдана**: це разова задача без поспіху, і звіт вийшов кориснішим за одиночну відповідь, бо містить конкретні застосунки й пріоритети для сімейного ChefBot. Для живого користувача, який стоїть біля плити й питає «чим замінити яйце», пів хвилини очікування **неприйнятні**. Там потрібен один агент з інструментом.

**Що змінив `share_member_interactions=False`?**
Запуск з `False` тривав 33,0 с проти 44,5 с, але дослідник зробив навіть більше пошуків (9 проти 7). Різниця в часі пояснюється радше збоєм 503 у першому запуску, ніж параметром. Структура звіту та сама. У другому звіті з'явився ще один застосунок (KptnCook). На одному запуску не можна сказати, що саме параметр спричинив ці відмінності: така ж різниця буває між двома однаковими запусками. Важливий висновок інший: **параметр не вимикає обмін інформацією**. Лідер усе одно передає аналітику результат дослідника в тексті завдання. З `True` учасники додатково бачать повні взаємодії одне одного, а з `False` — лише те, що лідер вирішив переказати.

**Неточна інформація від Researcher: чи є верифікація?**
Я навмисно передала Analyst три помилкові «факти». Він позначив усі три: мед дітям з 3 місяців назвав забороненим (ризик ботулізму до року), «склянку борошна 300 г» — помилкою (норма 130–160 г, «300 г — це приблизно дві склянки»), а «90 % користувачів» — твердженням без джерела. Отже, модель ловить помилки, які **суперечать загальновідомим знанням**. Окремого механізму верифікації в системі все одно **немає**: Analyst не має інструментів і перевіряє лише те, що «знає сам». Правдоподібну, але хибну деталь, наприклад яка саме програма вміє розпізнавати продукти з фото, він перевірити не може. Тож якість фінального звіту обмежена якістю пошуку, а впевнена структура аналітика робить неперевірені деталі переконливішими. Для продукту потрібен окремий крок перевірки: агент-верифікатор з пошуком, звірка з перевіреною базою або людина для безпекових тем.

## Завдання 4. Агент з reasoning

Агент з `ReasoningTools` (`think` + `analyze`). Питання з реальною дилемою для ChefBot: продуктове рішення з компромісами.

In [ ]:
from agno.tools.reasoning import ReasoningTools

REASONING_PROMPT = """
Я розробляю ChefBot — кулінарного AI-асистента для сім'ї.
Бюджет обмежений, я можу зробити лише одну функцію на наступні 2 місяці:

А) Розпізнавання продуктів з фото холодильника і підбір рецептів з того, що є.
Б) Тижневий планувальник меню з автоматичним списком покупок і врахуванням алергій членів сім'ї.

Користувачі — 3–5 осіб в одній сім'ї, готують щодня, один член сім'ї має алергію на горіхи.
Які фактори слід зважити (цінність, ризики, складність, точність AI)?
Що б ти обрав і чому? Що зробити, щоб зменшити ризики обраного варіанту?
"""

# Спільні інструкції для обох агентів (з reasoning і без), щоб різниця в результатах
# залежала лише від ReasoningTools, а не від тексту інструкцій (порада ментора).
COMMON_INSTRUCTIONS = [
    "Перед відповіддю розклади проблему на фактори і проаналізуй кожен покроково.",
    "Розглядай перспективи користувача, розробника і безпеки.",
    "Для кожного варіанту назви головний ризик.",
    "Чітко формулюй висновок з обґрунтуванням і умовами, за яких висновок змінився б.",
    "Відповідай українською мовою.",
]

reasoning_agent = Agent(
    name="ChefBot Product Advisor",
    db=InMemoryDb(),  # зберігаємо запуски в пам'яті, щоб потім прочитати токени й виклики інструментів
    model=make_model(),
    tools=[
        ReasoningTools(
            enable_think=True,      # інструмент «подумати» — чернетка міркувань
            enable_analyze=True,    # інструмент «проаналізувати» — оцінка проміжних висновків
            add_instructions=True,  # додає моделі інструкції, як користуватися цими кроками
        )
    ],
    instructions=COMMON_INSTRUCTIONS,
    markdown=True,
    stream_events=True,   # бачимо виклики think/analyze як окремі події
    debug_mode=True,
)
t4_reasoning = run_timed(reasoning_agent, REASONING_PROMPT)

DEBUG Agent initialized: chefbot-product-advisor

DEBUG Reading AgentSession: 717f8d0f-dd75-497d-b5ad-b2ca1c924c82

DEBUG Creating new AgentSession: 717f8d0f-dd75-497d-b5ad-b2ca1c924c82

DEBUG Agent Run Start: 0f257cb7-24dd-4ea9-8a83-a3ca1e371713

DEBUG Processing tools for model

DEBUG Added tool think from reasoning_tools

... (172 outputs left out)

DEBUG Created or updated AgentSession record: 717f8d0f-dd75-497d-b5ad-b2ca1c924c82

DEBUG Saved run 0f257cb7-24dd-4ea9-8a83-a3ca1e371713 to session 717f8d0f-dd75-497d-b5ad-b2ca1c924c82

DEBUG Agent Run End: 0f257cb7-24dd-4ea9-8a83-a3ca1e371713


⏱ Час виконання: 52.0 с


### Порівняння: той самий запит без ReasoningTools

In [ ]:
no_reasoning_agent = Agent(
    name="ChefBot Product Advisor (без reasoning)",
    db=InMemoryDb(),  # зберігаємо запуски в пам'яті, щоб потім прочитати токени й виклики інструментів
    model=make_model(),
    instructions=COMMON_INSTRUCTIONS,  # ті самі інструкції, що й у агента з reasoning
    markdown=True,
)
t4_plain = run_timed(no_reasoning_agent, REASONING_PROMPT)

# Порівнюємо токени і виклики інструментів останніх запусків
for name, ag, secs in [("з ReasoningTools", reasoning_agent, t4_reasoning),
                       ("без ReasoningTools", no_reasoning_agent, t4_plain)]:
    run = ag.get_last_run_output()
    tokens = run.metrics.total_tokens if run and run.metrics else "н/д"
    calls = [t.tool_name for t in (run.tools or [])] if run else []
    print(f"{name:20} | токени: {tokens} | час: {secs:.1f} с | інструменти: {calls}")

DEBUG Session ID: 1f09d878-f995-4d2d-9f0e-859645f20adb

Output()

ERROR    Error from Gemini API: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently       
         experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status':     
         'UNAVAILABLE'}}

WARNING  Model provider error during stream (attempt 1/4): <google.genai._api_client.HttpResponse object at        
         0x7e11b0861010>. : <google.genai._api_client.HttpResponse object at 0x7e11b0861010>Retrying in 10s...:    
         <google.genai._api_client.HttpResponse object at 0x7e11b0861010>


⏱ Час виконання: 20.5 с
з ReasoningTools     | токени: 25920 | час: 52.0 с | інструменти: ['think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'think', 'analyze', 'analyze']
без ReasoningTools   | токени: 1750 | час: 20.5 с | інструменти: []


> **Оновлення після рецензії ментора.** У першому запуску агент з ReasoningTools мав два додаткові рядки інструкцій («назви головний ризик», «умови, за яких висновок змінився б»), тож порівняння було не зовсім чистим. Тепер обидва агенти використовують спільний список `COMMON_INSTRUCTIONS`, і різниця залежить лише від ReasoningTools.
>
> Цифри в таблиці нижче (25 920 проти 1 750 токенів) — з попереднього, нечистого запуску. Після перезапуску комірок завдання 4 таблицю і висновки варто оновити.


### 📝 Продуктовий коментар до завдання 4

**Цифри запуску:**

| | З ReasoningTools | Без ReasoningTools |
|---|---|---|
| Токени | **25 920** | **1 750** |
| Час | 52,0 с | 20,5 с |
| Виклики інструментів | 9 × `think` + 2 × `analyze` | — |
| Висновок | Варіант Б | Варіант Б |

Час обох запусків роздутий повторами після помилок 503/429 (сервер Gemini був перевантажений), тож порівнювати варто насамперед токени: reasoning обійшовся приблизно у **15 разів** дорожче.

**Порівняння якості.** Обидва агенти дійшли до одного висновку (тижневий планувальник з урахуванням алергій) і дали структуровані відповіді. Інструкція «розклади на фактори» вже сама по собі робить відповідь покроковою. Відповідь з ReasoningTools сильніша в розділі про ризики. Вона прямо назвала **головний ризик обраного варіанта** (галюцинація моделі щодо алергену) і запропонувала конкретні захисти: програмний чорний список інгредієнтів, детерміновану перевірку кожного рецепта до показу, генерацію лише з перевіреної бази. Також вона назвала **умови, за яких висновок змінився б**. Чесне застереження: частину цієї різниці дають інструкції, бо агентові з reasoning я дала два додаткові рядки («назви головний ризик», «умови, за яких висновок змінився б»). Тобто порівняння не зовсім чисте.
У логах видно і зворотний бік: виклики `think` з назвами «Fixing think tool call» і «Corrected Think Call» показують, що модель частину токенів витратила на те, щоб правильно викликати сам інструмент міркувань. Reasoning **не завжди** покращує відповідь: за 15-кратну ціну ми отримали той самий висновок і кращий розділ про ризики.

**Де для ChefBot це виправдано, а де ні?**
*Виправдано:* тижневе меню з кількома обмеженнями одночасно (алергія, бюджет, час, що є в холодильнику), адаптація рецепта, коли заміни впливають одна на одну, розбір «чому не вдалася випічка» за кількома симптомами.
*Зайва витрата:* «скільки грамів у склянці цукру», «чим замінити розпушувач», «скільки варити яйце». Тут працює інструмент або коротка відповідь, і кожна зайва секунда погіршує досвід користувача біля плити.

**Де структурований аналіз допоміг, а де дав загальні міркування?**
*Допоміг:* у зв'язці «алергія на горіхи → головний ризик варіанта Б → детермінований фільтр, а не "чесне слово" LLM». Цей висновок прямо випливає з умов задачі і збігається з архітектурним принципом ChefBot: безпека має бути в коді, а не лише в промпті.
*Загальні міркування:* «Vision API дорожчі за текстові» без жодної цифри, «2 місяці — реально для однієї людини» без оцінки обсягу робіт, «варіант А підійшов би фуд-блогерам» без даних. Крім того, обидва агенти недооцінили безпекові ризики фото-варіанта: «помилка — це максимум зіпсована вечеря», хоча не помічена банка арахісової пасти для алергіка — так само загроза. Агент без reasoning у першій, перерваній збоєм, спробі оцінив той самий ризик як «вкрай небезпечно». Тобто навіть оцінки ризику не стабільні між запусками.
Висновок: ReasoningTools покращує **логіку поєднання наданих фактів**, але не додає **нових знань** і не гарантує правильних оцінок. Там, де фактів бракує, структура лише охайно оформлює загальні місця.

## Завдання 5. Адаптація ChefBot на Agno

Обраний інструмент: **`search_recipes`** з мого ChefBot теми 8 (LangChain, gpt-4o-mini). Це головний сценарій продукту: «що приготувати».
Перенесено **ту саму базу з 7 рецептів**, ту саму логіку пошуку за назвою, інгредієнтами й тегом, і додано обробку помилок, якої в темі 8 не було.
Тестові запити взято з таблиці тестування теми 8 (T1, T2, T7, T8, T9, T10, T11): це всі сценарії, що стосуються пошуку рецептів і меж продукту.

> Цей розділ можна запустити окремо: клацніть на першу комірку нижче → **Середовище виконання → Виконати після**.

In [ ]:
# Окреме налаштування, щоб Завдання 5 можна було запустити без попередніх розділів
%pip install --quiet "agno==3.1.12" "ddgs==9.16.1" "google-genai==2.23.0" "pytz==2026.6"

import os, json, time
try:
    from google.colab import userdata
    os.environ["GOOGLE_API_KEY"] = userdata.get("GOOGLE_API_KEY")
except Exception:
    import getpass
    os.environ["GOOGLE_API_KEY"] = getpass.getpass("Введіть Google Gemini API key: ")

from agno.agent import Agent
from agno.models.google import Gemini
from agno.db.in_memory import InMemoryDb
from agno.tools import tool

MODEL_ID = "gemini-3.5-flash-lite"


def make_model() -> Gemini:
    """Модель з автоматичними повторами при тимчасових збоях."""
    return Gemini(id=MODEL_ID, retries=3, delay_between_retries=10, exponential_backoff=True)


print("✅ Налаштування для Завдання 5 готові")

✅ Налаштування для Завдання 5 готові


In [ ]:
# База рецептів — перенесена з теми 8 без змін
RECIPES = [
    {
        "name": "Сирники класичні",
        "tags": ["сніданок", "вегетаріанське"],
        "time_min": 25, "servings": 2,
        "ingredients": {"сир кисломолочний": "400 г", "яйце": "1 шт", "борошно": "3 ст. л.",
                        "цукор": "2 ст. л.", "сіль": "дрібка", "олія": "для смаження"},
        "steps": ["Розім'яти сир виделкою.", "Додати яйце, цукор, сіль, перемішати.",
                  "Всипати борошно, сформувати сирники.", "Смажити на середньому вогні по 3–4 хв з кожного боку."],
    },
    {
        "name": "Омлет з овочами",
        "tags": ["сніданок", "швидко", "без глютену"],
        "time_min": 15, "servings": 1,
        "ingredients": {"яйце": "3 шт", "молоко": "50 мл", "помідор": "1 шт",
                        "перець болгарський": "0.5 шт", "сіль": "за смаком", "вершкове масло": "10 г"},
        "steps": ["Нарізати овочі.", "Збити яйця з молоком і сіллю.",
                  "Обсмажити овочі на маслі 2 хв.", "Залити яєчною сумішшю, готувати під кришкою 5–6 хв."],
    },
    {
        "name": "Борщ український",
        "tags": ["обід", "суп"],
        "time_min": 120, "servings": 6,
        "ingredients": {"буряк": "2 шт", "капуста": "300 г", "картопля": "3 шт", "морква": "1 шт",
                        "цибуля": "1 шт", "томатна паста": "2 ст. л.", "свинина": "500 г",
                        "часник": "2 зубчики", "сіль": "за смаком"},
        "steps": ["Зварити бульйон зі свинини (1–1.5 год).", "Додати нарізану картоплю.",
                  "Окремо потушкувати буряк, моркву, цибулю з томатною пастою.",
                  "Додати капусту, через 5 хв — засмажку.", "Довести до кипіння, додати часник, дати настоятися 20 хв."],
    },
    {
        "name": "Паста з томатним соусом",
        "tags": ["обід", "вечеря", "вегетаріанське", "швидко"],
        "time_min": 20, "servings": 2,
        "ingredients": {"паста": "200 г", "помідор": "3 шт", "часник": "2 зубчики",
                        "оливкова олія": "2 ст. л.", "базилік": "кілька листків", "сіль": "за смаком"},
        "steps": ["Відварити пасту al dente.", "Обсмажити часник на олії 30 с.",
                  "Додати нарізані помідори, тушкувати 8–10 хв.", "Змішати з пастою, додати базилік."],
    },
    {
        "name": "Курячий суп з локшиною",
        "tags": ["обід", "суп"],
        "time_min": 50, "servings": 4,
        "ingredients": {"курка": "400 г", "локшина": "100 г", "картопля": "2 шт",
                        "морква": "1 шт", "цибуля": "1 шт", "сіль": "за смаком"},
        "steps": ["Зварити бульйон із курки (30 хв).", "Додати картоплю й моркву, варити 10 хв.",
                  "Додати локшину, варити ще 5–7 хв.", "Посолити, подавати із зеленню."],
    },
    {
        "name": "Млинці на молоці",
        "tags": ["сніданок", "десерт", "вегетаріанське"],
        "time_min": 40, "servings": 4,
        "ingredients": {"молоко": "500 мл", "яйце": "2 шт", "борошно": "200 г",
                        "цукор": "1 ст. л.", "сіль": "дрібка", "олія": "2 ст. л."},
        "steps": ["Збити яйця з цукром і сіллю.", "Додати половину молока і борошно, вимісити без грудочок.",
                  "Влити решту молока й олію.", "Смажити на розігрітій сковорідці по 1 хв з кожного боку."],
    },
    {
        "name": "Овочеве рагу",
        "tags": ["вечеря", "вегетаріанське", "веганське", "без глютену"],
        "time_min": 40, "servings": 3,
        "ingredients": {"кабачок": "1 шт", "картопля": "3 шт", "морква": "1 шт", "цибуля": "1 шт",
                        "перець болгарський": "1 шт", "помідор": "2 шт", "олія": "2 ст. л.", "сіль": "за смаком"},
        "steps": ["Нарізати овочі кубиками.", "Обсмажити цибулю й моркву.",
                  "Додати картоплю, тушкувати 10 хв.", "Додати решту овочів, тушкувати ще 15–20 хв."],
    },
]

ALLOWED_TAGS = sorted({t for r in RECIPES for t in r["tags"]})


def _norm(s: str) -> str:
    return s.lower().strip()


@tool
def search_recipes(query: str = "", ingredients: list[str] | None = None, tag: str = "") -> str:
    """Шукає рецепти у перевіреній базі ChefBot за назвою страви, наявними продуктами або категорією.

    Використовуй ЗАВЖДИ, коли користувач просить рецепт, питає «що приготувати»
    або перелічує продукти, які має. Подавай користувачу лише те, що повернув інструмент:
    не додавай пропорцій чи кроків від себе.
    НЕ використовуй для питань не про їжу, для медичних дієт і для переведення мір.

    Args:
        query: Назва страви або її частина українською ("борщ", "сирники"). Може бути порожнім,
            якщо задано ingredients або tag.
        ingredients: Список продуктів, які є у користувача, у називному відмінку
            (["яйце", "помідор", "перець"]).
        tag: Категорія: "сніданок", "обід", "вечеря", "суп", "швидко", "вегетаріанське",
            "веганське", "без глютену", "десерт". Може бути порожнім.

    Returns:
        JSON: {"found": N, "recipes": [...]} — до 3 рецептів з інгредієнтами, кроками
        і полем missing_ingredients (чого бракує), або {"found": 0, "message": ...},
        або {"error": ...} з поясненням, що не так із запитом.
    """
    # --- Обробка помилок (нове порівняно з темою 8) ---
    ingredients = [_norm(i) for i in (ingredients or []) if i and i.strip()]
    if not query.strip() and not ingredients and not tag.strip():
        return json.dumps({"error": "Порожній запит: вкажи назву страви, продукти або категорію."},
                          ensure_ascii=False)
    if tag and _norm(tag) not in ALLOWED_TAGS:
        return json.dumps({"error": f"Невідома категорія '{tag}'. Доступні: {', '.join(ALLOWED_TAGS)}."},
                          ensure_ascii=False)

    # --- Логіка пошуку з теми 8 ---
    results = []
    for r in RECIPES:
        if query and not all(w[:4] in _norm(r["name"]) for w in _norm(query).split()):
            continue
        if tag and _norm(tag) not in r["tags"]:
            continue
        score = 0
        if ingredients:
            names = [_norm(k) for k in r["ingredients"]]
            score = sum(any(i[:3] in n for n in names) for i in ingredients)
            if score == 0:
                continue
        results.append((score, r))

    if not results:
        return json.dumps({"found": 0,
                           "message": "У базі немає рецептів за цим запитом.",
                           "available": [r["name"] for r in RECIPES]}, ensure_ascii=False)

    results.sort(key=lambda x: -x[0])
    out = []
    for score, r in results[:3]:
        item = dict(r)
        if ingredients:
            item["matched_ingredients"] = score
            item["missing_ingredients"] = [k for k in r["ingredients"]
                                           if not any(i[:3] in _norm(k) for i in ingredients)
                                           and k not in ("сіль", "олія")]
        out.append(item)
    return json.dumps({"found": len(out), "recipes": out}, ensure_ascii=False)

### Юніт-тести інструменту (без моделі)

In [ ]:
# Ті самі виклики, що й у модульній перевірці теми 8, плюс нові випадки помилок
for kwargs in [
    {"query": "борщ"},
    {"ingredients": ["яйце", "помідор", "перець"]},
    {"tag": "суп"},
    {"query": "том ям"},
    {},                                        # порожній запит
    {"tag": "кето"},                           # невідома категорія
    {"ingredients": ["  ", ""]},               # порожні елементи списку
]:
    res = json.loads(search_recipes.entrypoint(**kwargs))
    summary = res.get("error") or res.get("message") or [r["name"] for r in res["recipes"]]
    print(f"{str(kwargs):45} → {summary}")

{'query': 'борщ'}                             → ['Борщ український']
{'ingredients': ['яйце', 'помідор', 'перець']} → ['Омлет з овочами', 'Овочеве рагу', 'Сирники класичні']
{'tag': 'суп'}                                → ['Борщ український', 'Курячий суп з локшиною']
{'query': 'том ям'}                           → У базі немає рецептів за цим запитом.
{}                                            → Порожній запит: вкажи назву страви, продукти або категорію.
{'tag': 'кето'}                               → Невідома категорія 'кето'. Доступні: без глютену, веганське, вегетаріанське, вечеря, десерт, обід, сніданок, суп, швидко.
{'ingredients': ['  ', '']}                   → Порожній запит: вкажи назву страви, продукти або категорію.


In [ ]:
# System prompt з теми 8, адаптований: у цій версії лише один інструмент
chefbot = Agent(
    name="ChefBot",
    db=InMemoryDb(),  # зберігаємо запуски, щоб прочитати виклики інструментів
    model=make_model(),
    tools=[search_recipes],
    instructions=[
        "Ти — ChefBot, дружній і точний кулінарний асистент для домашньої кухні. "
        "Відповідай українською, коротко і по ділу, тоном досвідченої господині.",
        "search_recipes — для БУДЬ-ЯКОГО запиту на рецепт або «що приготувати з…». "
        "Якщо користувач перелічує продукти, передай їх у параметр ingredients.",
        "Подавай лише рецепти, які повернув інструмент. НЕ вигадуй рецептів і пропорцій.",
        "Якщо інструмент нічого не знайшов — чесно скажи, що в базі такого рецепта немає, "
        "і запропонуй найближчу альтернативу САМЕ з бази (поле available).",
        "Покажи, яких інгредієнтів бракує (поле missing_ingredients).",
        "Інструмента замінників у цій версії немає: якщо потрібна заміна продукту, "
        "дай коротку загальну пораду і прямо познач, що вона не з перевіреної бази.",
        "Ти відповідаєш ЛИШЕ на кулінарні теми; на інші ввічливо відмовляй і повертай розмову до кухні.",
        "Ти НЕ дієтолог і НЕ лікар: не складай лікувальних дієт, порадь звернутися до фахівця.",
        "Якщо запит неоднозначний — постав одне коротке уточнювальне питання.",
    ],
    markdown=True,
    debug_mode=True,
)

### Тестові сценарії з таблиці тестування теми 8

In [ ]:
import pandas as pd

# (ID, сценарій, запит, інструменти в темі 8, початок відповіді в темі 8) — з мого ноутбука теми 8
TEST_CASES = [
    ("T1", "Пошук рецепта", "Дай рецепт борщу",
     "search_recipes", "Рецепт борщу з бази: 6 порцій, інгредієнти й кроки без змін"),
    ("T2", "Пошук за інгредієнтами", "У мене є яйця, помідор і перець. Що приготувати?",
     "search_recipes", "Омлет з овочами, овочеве рагу, сирники + чого бракує"),
    ("T7", "Ланцюжок інструментів", "Хочу сирники, але без яєць. Що робити?",
     "find_substitute (без search_recipes ⚠️)", "3 замінники яйця, рецепт сирників не знайдено"),
    ("T8", "Рецепта немає в базі", "Дай рецепт тайського супу том ям",
     "search_recipes", "У базі немає; запропонував борщ або «грибний суп» (якого в базі немає)"),
    ("T9", "Поза межами (тема)", "Яка завтра буде погода у Львові?",
     "—", "Ввічлива відмова, повернення до кухні"),
    ("T10", "Поза межами (медицина)", "Склади мені лікувальну дієту від гастриту на тиждень",
     "—", "Відмова від медичних порад, пропозиція рецептів"),
    ("T11", "Неоднозначний запит", "Скільки це в грамах?",
     "—", "Уточнювальне питання"),
]

rows = []
for tid, scenario, query, lc_tools, lc_answer in TEST_CASES:
    print("=" * 100)
    print(f"🧪 {tid} · {scenario}: {query}")
    time.sleep(6)  # пауза, щоб не перевищити ліміт запитів за хвилину
    start = time.perf_counter()
    chefbot.print_response(query, stream=True)
    elapsed = time.perf_counter() - start
    run = chefbot.get_last_run_output()
    agno_tools = [t.tool_name for t in (run.tools or [])] if run else []
    answer = (run.content or "") if run else ""
    rows.append({
        "ID": tid, "Сценарій": scenario,
        "LangChain: інструменти": lc_tools,
        "Agno: інструменти": ", ".join(agno_tools) or "—",
        "LangChain: відповідь": lc_answer,
        "Agno: відповідь (початок)": answer[:150].replace("\n", " ") + "…",
        "Agno: час, с": round(elapsed, 1),
    })

🧪 T1 · Пошук рецепта: Дай рецепт борщу


DEBUG Agent initialized: chefbot

DEBUG Reading AgentSession: cd6706b0-51a6-49ab-9a21-a1f908f80348

DEBUG Creating new AgentSession: cd6706b0-51a6-49ab-9a21-a1f908f80348

DEBUG Agent Run Start: 6903e77f-0f3b-450b-a025-31808eb74b2c

DEBUG Processing tools for model

... (224 outputs left out)

DEBUG Added RunOutput to Agent Session

DEBUG Created or updated AgentSession record: cd6706b0-51a6-49ab-9a21-a1f908f80348

DEBUG Saved run 33912aa7-cf3c-4511-a54c-68f8f9b45283 to session cd6706b0-51a6-49ab-9a21-a1f908f80348

DEBUG Agent Run End: 33912aa7-cf3c-4511-a54c-68f8f9b45283

In [ ]:
# Зведена таблиця: тема 8 (LangChain) проти теми 11 (Agno)
pd.set_option("display.max_colwidth", 160)
comparison = pd.DataFrame(rows)
comparison

,ID,Сценарій,LangChain: інструменти,Agno: інструменти,LangChain: відповідь,Agno: відповідь (початок),"Agno: час, с"
0,T1,Пошук рецепта,search_recipes,search_recipes,"Рецепт борщу з бази: 6 порцій, інгредієнти й кроки без змін","Ось перевірений рецепт українського борщу з нашої бази: ### Борщ український * **Час приготування:** 120 хв * **Порції:** 6 * **Категорія:** обід, с…",2.4
1,T2,Пошук за інгредієнтами,search_recipes,search_recipes,"Омлет з овочами, овочеве рагу, сирники + чого бракує","Із ваших продуктів чудово вийде **Омлет з овочами**! Ось рецепт з нашої бази: **Омлет з овочами** *(Час: 15 хв, Порцій: 1)* * **Інгредієнти:** яй…",2.1
2,T7,Ланцюжок інструментів,find_substitute (без search_recipes ⚠️),search_recipes,"3 замінники яйця, рецепт сирників не знайдено","У нашій перевіреній базі є класичний рецепт сирників, але в ньому використовується яйце. Ось що входить до складу страви **«Сирники класичні»** (час…",2.5
3,T8,Рецепта немає в базі,search_recipes,search_recipes,У базі немає; запропонував борщ або «грибний суп» (якого в базі немає),У моїй базі перевірених рецептів «Том Ям» наразі немає. Але можу запропонувати інші перші страви з нашої бази: - Борщ український - Курячий суп з ло…,1.4
4,T9,Поза межами (тема),—,—,"Ввічлива відмова, повернення до кухні","Я допомагаю лише з кулінарними питаннями! Розкажіть краще, що у вас є в холодильнику, а я підберу рецепт для смачного обіду чи вечері.…",0.6
5,T10,Поза межами (медицина),—,—,"Відмова від медичних порад, пропозиція рецептів","Я не лікар і не дієтолог, тому не можу складати лікувальні дієти. При гастриті обов'язково проконсультуйтеся з гастроентерологом або лікарем-дієтиком!…",0.9
6,T11,Неоднозначний запит,—,—,Уточнювальне питання,"Схоже, ви запитуєте про переведення мір, але не вказали, що саме треба перевести! Будь ласка, напишіть, який продукт і скільки ложках/склянках вимір…",0.9


### Код інструменту в темі 8 (LangChain), для порівняння

```python
from langchain_core.tools import tool

@tool
def search_recipes(query: str = "", ingredients: list[str] | None = None, tag: str = "") -> str:
    """Шукає рецепти у перевіреній базі ChefBot.
    Використовуй, коли користувач просить рецепт, питає «що приготувати»
    або перелічує продукти, які має. ..."""
    ...  # та сама логіка пошуку, 45 рядків разом із декоратором

agent = create_agent(model=llm, tools=tools, system_prompt=SYSTEM_PROMPT)

def used_tools(messages):          # власна функція, щоб дістати виклики інструментів
    ...
def ask_once(question):            # власна обгортка для одного запиту
    result = agent.invoke({"messages": [HumanMessage(content=question)]})
    ...
```

### Порівняльна таблиця

| Критерій | LangChain (тема 8) | Agno (тема 11) |
|---|---|---|
| Кількість рядків коду для одного tool | 45 рядків (декоратор `@tool` + функція `search_recipes`) | 62 рядки: та сама логіка + довший docstring + 7 рядків обробки помилок (порожній запит, невідома категорія). Сам декоратор `@tool` і спосіб опису ідентичні |
| Обв'язка навколо tool | `create_agent(...)` + власні функції `used_tools()` і `ask_once()` (~20 рядків), щоб дістати з повідомлень список викликів | `Agent(..., tools=[search_recipes])`; виклики й токени є «з коробки» в `get_last_run_output()` та в логах `debug_mode` |
| Чіткість docstring | Добра: є «коли використовувати», параметри й формат відповіді | Краща: додано «НЕ використовуй для…», заборону додавати пропорції від себе, опис поля помилки `error` |
| Зрозумілість для нового розробника | Треба розуміти структуру повідомлень (`AIMessage.tool_calls`), щоб побачити, що робив агент | Агент читається як конфігурація; що робив агент, видно в журналі без додаткового коду |
| Поведінка агента на тестових запитах | 6 з 7 сценаріїв ✅; T7 ⚠️ (не викликав `search_recipes`); T8 запропонував «грибний суп», якого в базі немає | 7 з 7 сценаріїв викликали очікуваний інструмент; T8 запропонував альтернативи лише з бази. T2 показав 1 рецепт замість 3 |
| Де виникли проблеми при перенесенні | — | System prompt перенесено майже дослівно, але правило про замінники довелося переписати, бо `find_substitute` не переносився. Gemini передав «яйця» замість «яйце» (пошук спрацював завдяки порівнянню за першими літерами). Довелося додати повтори при помилках 503/429 і паузи між запитами, бо безкоштовний ліміт Gemini значно жорсткіший |

### 📝 Продуктовий коментар до завдання 5

**Порівняння результатів на тих самих тестових запитах (таблиця `comparison` вище):**
- **T1 «рецепт борщу»:** обидві версії викликали `search_recipes` і подали рецепт із бази без змін.
- **T2 «яйця, помідор, перець»:** LangChain показав три рецепти з переліком того, чого бракує. Agno показав лише омлет, теж з полем «бракує», і додав пораду щодо заміни молока з позначкою «не з перевіреної бази». Інструмент в Agno повернув ті самі три рецепти, але модель вирішила показати один.
- **T7 «сирники без яєць»:** тут Agno кращий. У темі 8 агент одразу пішов до замінників і не знайшов рецепт, а в Agno викликав `search_recipes`, показав рецепт з бази й окремо позначену пораду щодо заміни яйця.
- **T8 «том ям»:** в обох випадках «у базі немає». LangChain запропонував «грибний суп», якого в базі немає, а Agno — лише борщ і курячий суп з бази (поле `available`).
- **T9–T11:** однакова правильна поведінка: відмова поза темою, відмова від лікувальної дієти з порадою звернутися до лікаря, уточнювальне питання.

**Що було найскладнішим при перенесенні: технічне чи концептуальне?**
Технічно перенесення майже тривіальне: декоратор `@tool` в обох фреймворках однаковий, і функція з базою переїхала без змін. Технічні проблеми були не у фреймворку, а в **провайдері моделі**: ліміт 20 запитів на день для gemini-3.8-flash, помилки 503, нестабільний DuckDuckGo. Найскладнішим було **концептуальне**: вирішити, що робити з частиною логіки, яка в темі 8 жила в інших інструментах (`find_substitute`), і як чесно розмежувати «з бази» та «загальна порада» в одному інструменті.

**Чи змінилася поведінка на тих самих запитах? Чому?**
Так, у трьох сценаріях з семи (T2, T7, T8). Причина не у фреймворку: обидва фреймворки передають моделі той самий опис функції. Змінилися три речі: **модель** (gpt-4o-mini → gemini-3.5-flash-lite), **docstring і відповідь інструмента** (поле `available` прямо дало моделі альтернативи з бази, тому зник вигаданий «грибний суп») і **набір інструментів** (без `find_substitute` модель у T7 мусила спершу знайти рецепт). Висновок: фреймворк визначає зручність розробки й спостережуваність, а **поведінку визначають модель, промпт, опис інструментів і дані, які інструмент повертає**.

**LangChain чи Agno для production-версії ChefBot?**
Я б обґрунтувала перед командою вибір **Agno**:
1. **Спостережуваність без додаткового коду.** У темі 8 я писала власні `used_tools()` і `ask_once()`, щоб побачити виклики інструментів. В Agno `debug_mode` і `get_last_run_output()` одразу показують виклики, аргументи й токени. Саме завдяки цьому я помітила неправильну атрибуцію джерел у завданні 1 і 15-кратну ціну reasoning у завданні 4.
2. **Вбудована стійкість.** Параметри `retries` і `exponential_backoff` у моделі дозволили ноутбуку пройти крізь помилки 503/429 без зміни коду. Для продукту з безкоштовним або дешевим тарифом це критично.
3. **Готові мультиагентні режими.** `Team` у режимі `coordinate` з делегуванням працює «з коробки» (завдання 3). Наступний крок ChefBot, розділення на спеціалістів з рецептів, замінників і планування меню, не потребуватиме переписування архітектури.

*Застереження:* у LangChain значно більша екосистема (готові інтеграції з векторними базами, LangSmith для систематичного оцінювання, більше прикладів і відповідей у спільноті). Якщо ChefBot виросте до великої бази рецептів із семантичним пошуком (RAG), цей аргумент може переважити, і рішення варто переглянути після прототипу.

---
### Підсумок
- Завдання 1: агент з веб-пошуком, 3 запити + контрольний агент без інструменту.
- Завдання 2: 2 власні інструменти (`get_current_datetime`, `convert_kitchen_measure`) з юніт-тестами і 5 тестами агента.
- Завдання 3: команда Researcher + Analyst у режимі coordinate, порівняння `share_member_interactions`, тест на неточні дані.
- Завдання 4: агент з ReasoningTools і без них, порівняння токенів, часу й викликів.
- Завдання 5: `search_recipes` з теми 8, перенесений на Agno з обробкою помилок; 7 тестових сценаріїв теми 8 і порівняльна таблиця LangChain vs Agno.